In [ ]:
# ============================================================
# CELL 0 — IMPORTS & USER CONFIGURATION
# ============================================================
# This notebook requires the main pipeline to have been run through Step 1.
# The following objects must already exist in your kernel session:
#   df_long, compute_premeans, PRE_AVG_YRS, PRE_MEAN_VARS, STATIC_COVARS
#   pd, np, sm, StandardScaler, LogisticRegression, variance_inflation_factor

# add the two missing imports 
from itertools import combinations
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# ── USER CONFIGURATION ────────────────────────────────────────────────────────
#  Define ALL_CANDIDATES here before running.
# List every pixel-level column you want to screen as a potential confounder.

ALL_CANDIDATES = [
    # add or remove candidates here based on your dataset
]



# Maximum search pool size before the exhaustive search is skipped.
# 2^n subsets: n=12 → ~4k fits (fast); n=15 → ~32k fits (slow); n=18 → hours.
MAX_SEARCH_POOL = 13

print('✓ Imports and configuration loaded')
print(f'  Candidates defined: {len(ALL_CANDIDATES)}')



---
##  Confounder selection

This step:
1. Computes pre-treatment means for all candidates
2. Flags known problem variables
3. Runs VIF screening
4. Runs overlap-guided automatic selection (AUC 0.65–0.85, overlap > 20%, mean |SMD| < 0.25)
5. Prints the recommended final set

**The selection of covariates are highly expert dependent!!!.

In [ ]:
# ── Compute pre-treatment means ───────────────────────────────────────────────
print(f'Computing {PRE_AVG_YRS}-year pre-treatment means...')
df_cs_full = compute_premeans(df_long, PRE_MEAN_VARS)

# All candidates that exist in df_cs_full
avail_all = [c for c in ALL_CANDIDATES if c in df_cs_full.columns]
missing   = [c for c in ALL_CANDIDATES if c not in df_cs_full.columns]

print(f'\nCandidates found ({len(avail_all)}): {avail_all}')
if missing:
    print(f'Not in data ({len(missing)}): {missing}')



In [ ]:
# ── Within-reach variance check ───────────────────────────────────────────────
# Variables with near-zero within-reach SD are reach constants, not pixel confounders
df_cs_full_clean = df_cs_full.dropna(subset=avail_all, how='all').copy()

print(f'{"Variable":<30}{"Within-reach SD":>18}{"Total SD":>10}{"Ratio":>8}  Verdict')
print('-'*80)
REACH_CONSTANTS = []
for cov in avail_all:
    if cov not in df_cs_full_clean.columns: continue
    total_sd  = df_cs_full_clean[cov].std()
    within_sd = df_cs_full_clean.groupby('reach_numb')[cov].std().fillna(0).mean()
    if total_sd == 0: continue
    ratio = within_sd / total_sd
    if within_sd < 0.01:
        verdict = '✗ REACH CONSTANT — will destroy overlap'
        REACH_CONSTANTS.append(cov)
    elif ratio < 0.2:
        verdict = '⚠  mostly between-reach — risky'
    else:
        verdict = '✓ pixel-level variation'
    print(f'{cov:<30}{within_sd:>18.4f}{total_sd:>10.4f}{ratio:>8.3f}  {verdict}')

print(f'\nReach constants (auto-excluded from selection): {REACH_CONSTANTS}')

## Overlap-guided automatic selection

In [ ]:
# ── Overlap-guided automatic selection ───────────────────────────────────────
# Criteria: AUC 0.65–0.85 | overlap > 20% | mean|SMD| < 0.25
# Reach constants and known mediators/instruments excluded from candidate pool.

EXCLUDE_FROM_SEARCH = set(REACH_CONSTANTS) | KNOWN_MEDIATORS_INSTRUMENTS
SEARCH_POOL = [c for c in avail_all if c not in EXCLUDE_FROM_SEARCH]
df_cs_search = df_cs_full_clean.copy()

n_subsets = 2**len(SEARCH_POOL) - 1
print(f'Search pool ({len(SEARCH_POOL)}): {SEARCH_POOL}')
print(f'Excluded from search: {EXCLUDE_FROM_SEARCH & set(avail_all)}')
print(f'Subsets to evaluate: {n_subsets:,}')

# FIX 6: guard against combinatorial explosion before starting
if len(SEARCH_POOL) > MAX_SEARCH_POOL:
    raise ValueError(
        f'Search pool has {len(SEARCH_POOL)} variables ({n_subsets:,} subsets). '
        f'This exceeds MAX_SEARCH_POOL={MAX_SEARCH_POOL} and would take too long.\n'
        f'Manually remove variables from ALL_CANDIDATES or raise MAX_SEARCH_POOL '
        f'only if you are prepared to wait (each step roughly doubles runtime).'
    )

print(f'\nEvaluating {n_subsets:,} subsets...')


def eval_set(df_cs, conf_set):
    av = [c for c in conf_set if c in df_cs.columns]
    df_ = df_cs.dropna(subset=av).copy()
    if len(df_) < 10 or df_['is_treated'].nunique() < 2:
        return None
    X   = StandardScaler().fit_transform(df_[av].astype(float))
    mdl = LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs', random_state=42)
    mdl.fit(X, df_['is_treated'].values)
    ps  = mdl.predict_proba(X)[:, 1]
    auc = roc_auc_score(df_['is_treated'].values, ps)

    t_ps = ps[df_['is_treated'].values == 1]
    c_ps = ps[df_['is_treated'].values == 0]

    # FIX 4: replace the PS range-length proxy with the fraction of TREATED
    # units whose PS falls within the min–max range of CONTROL units.
    # This is the standard "common support" overlap check:
    #   a treated unit is "in support" if c_ps.min() <= ps_i <= c_ps.max().
    # Reported as a percentage of treated units.
    c_min, c_max = c_ps.min(), c_ps.max()
    in_support = ((t_ps >= c_min) & (t_ps <= c_max)).mean() * 100  # % of treated
    ov = in_support

    p_a = df_['is_treated'].mean()
    sw  = np.where(df_['is_treated'] == 1, p_a / ps, (1 - p_a) / (1 - ps))
    sw  = np.clip(sw, 0, np.quantile(sw, 0.99))
    df_['sw_tmp'] = sw

    t_ = df_[df_['is_treated'] == 1]
    c_ = df_[df_['is_treated'] == 0]
    smds = []
    for cov in av:
        tm = np.average(t_[cov], weights=t_['sw_tmp'])
        cm = np.average(c_[cov], weights=c_['sw_tmp'])
        tv = np.average((t_[cov] - tm)**2, weights=t_['sw_tmp'])
        cv = np.average((c_[cov] - cm)**2, weights=c_['sw_tmp'])
        sd = np.sqrt((tv + cv) / 2)
        smds.append(abs(tm - cm) / sd if sd > 0 else 0)

    return {
        'conf': av, 'n': len(av), 'auc': auc,
        'overlap': ov,   # % of treated units in control PS support
        'mean_smd': np.mean(smds) if smds else 1.0
    }


results = []
for sz in range(1, len(SEARCH_POOL) + 1):
    for combo in combinations(SEARCH_POOL, sz):
        r = eval_set(df_cs_search, list(combo))
        if r:
            results.append(r)

res_df = pd.DataFrame([
    {'conf': r['conf'], 'n': r['n'], 'auc': r['auc'],
     'overlap': r['overlap'], 'mean_smd': r['mean_smd']}
    for r in results
])

good = res_df[
    (res_df['auc'] >= 0.65) & (res_df['auc'] <= 0.85) &
    (res_df['overlap'] >= 20.0) &     # >= 20% of treated units in control support
    (res_df['mean_smd'] <= 0.25)
].sort_values(['mean_smd', 'overlap'], ascending=[True, False])

print(f'Subsets evaluated: {len(res_df):,} | Meeting criteria: {len(good)}')
print()
print(f'{"Rank":<6}{"AUC":>7}{"Overlap%":>10}{"Mean|SMD|":>11}  Confounders')
print('-'*80)
for i, (_, row) in enumerate(good.head(10).iterrows()):
    print(f'{i+1:<6}{row["auc"]:>7.3f}{row["overlap"]:>10.1f}{row["mean_smd"]:>11.4f}  {row["conf"]}')
